# Case Study 2 — Credit Card Fraud Detection (XGBoost): train · evaluate · submit

Dataset: Kaggle [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) — 284,807 European cardholder transactions (September 2013), of which only **492 (0.173%) are frauds**: a heavily imbalanced binary classification problem.

The pipeline is identical in shape to case study 1, with three adaptations for imbalanced tree-based learning:

- **XGBoost** is the default estimator
- numeric features are median-imputed but **not scaled** — tree models are invariant to monotone rescaling
- imbalance is handled with `scale_pos_weight = n_negative / n_positive`, and **average precision (PR-AUC)** joins the metrics, because plain accuracy is meaningless here (predicting "not fraud" for every row already scores 99.83%)

**How to use in Google Colab:** `Runtime → Run all`. The full dataset (zipped, 68.5 MB — the raw csv exceeds GitHub's 100 MB limit) is downloaded automatically from the project's GitHub repo and read directly by pandas; fallbacks: Kaggle (`kagglehub`) then a manual upload prompt.


In [ ]:
import re
import sys
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from xgboost import XGBClassifier, XGBRegressor
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=True)
    from xgboost import XGBClassifier, XGBRegressor

RANDOM_STATE = 42
print("pandas", pd.__version__)


In [ ]:
# ---- configuration: everything dataset-specific lives here ----
TRAIN_CSV = "credit_card_fraud.csv.zip"   # pandas reads the single-file zip directly
TEST_CSV = None            # set to the graded test csv name once released
TARGET = "Class"           # 1 = fraud, 0 = legitimate
ID_COLUMN = None           # dataset has no transaction id - submissions are keyed by row index
TEST_SIZE = 0.40           # holdout fraction used for evaluation
MODEL = None               # None = auto: xgboost (classification) or linear (regression)
SUBMISSION_OUT = "submission.csv"

# Experiment only: predict on the training file itself. In-sample predictions are
# NOT a valid submission.
DEMO_SUBMISSION_ON_TRAIN = False

DATA_URL_BASE = "https://raw.githubusercontent.com/zenpieV/mle_ca1_alok_29/main/"
KAGGLE_DATASET = "mlg-ulb/creditcardfraud"


In [ ]:
from pathlib import Path

def load_csv(name):
    # local file first, then the GitHub raw copy, then Kaggle, then a manual upload prompt in Colab
    if Path(name).exists():
        return pd.read_csv(name)
    try:
        frame = pd.read_csv(DATA_URL_BASE + Path(name).name)
        print(f"Downloaded '{name}' from {DATA_URL_BASE}")
        return frame
    except Exception:
        pass
    try:
        import kagglehub
        folder = kagglehub.dataset_download(KAGGLE_DATASET)
        candidates = sorted(Path(folder).rglob("*.csv"))
        if candidates:
            print(f"Downloaded '{candidates[0].name}' from Kaggle dataset {KAGGLE_DATASET}")
            return pd.read_csv(candidates[0])
    except Exception:
        pass
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"'{name}' not found locally, on GitHub, or on Kaggle")
    print(f"Could not find or download '{name}' - please upload it:")
    uploaded = files.upload()
    return pd.read_csv(next(iter(uploaded)))


## Preprocessing and model

All 30 features are numeric (PCA components `V1`–`V28` plus `Time` and `Amount`), so the categorical branch of the preprocessor is unused. Numeric features are median-imputed but **not** scaled: unlike case study 1's logistic regression, XGBoost splits on thresholds and is invariant to monotone rescaling. `scale_pos_weight` is computed from the training split to compensate for the 1:577 class imbalance.


In [ ]:
def detect_task(y):
    # text, boolean or categorical targets -> classification; numeric -> regression
    if (
        pd.api.types.is_bool_dtype(y)
        or pd.api.types.is_string_dtype(y)
        or pd.api.types.is_object_dtype(y)
        or isinstance(y.dtype, pd.CategoricalDtype)
    ):
        return "classification"
    if pd.api.types.is_numeric_dtype(y) and y.nunique() == 2:
        # a two-valued numeric column (e.g. the 0/1 fraud label) is classification
        return "classification"
    return "regression"

def build_pipeline(task, model_name, X, y=None):
    numeric_features = X.select_dtypes(include="number").columns.tolist()
    categorical_features = X.select_dtypes(exclude="number").columns.tolist()

    numeric_pipeline = Pipeline([
        # no scaler: tree-based models are invariant to monotone rescaling
        ("imputer", SimpleImputer(strategy="median")),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ])

    scale_pos_weight = None
    if task == "classification" and y is not None:
        classes = sorted(pd.unique(y), key=str)
        if len(classes) == 2:
            positive = classes[-1]
            scale_pos_weight = float((y != positive).sum() / (y == positive).sum())

    estimators = {
        ("classification", "xgboost"): lambda: XGBClassifier(
            n_estimators=300, learning_rate=0.1, max_depth=6, tree_method="hist",
            n_jobs=-1, random_state=RANDOM_STATE, eval_metric="aucpr",
            scale_pos_weight=scale_pos_weight,
        ),
        ("classification", "logistic"): lambda: LogisticRegression(
            max_iter=5000, random_state=RANDOM_STATE
        ),
        ("classification", "random_forest"): lambda: RandomForestClassifier(
            n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
        ),
        ("regression", "xgboost"): lambda: XGBRegressor(
            n_estimators=300, learning_rate=0.1, max_depth=6, tree_method="hist",
            n_jobs=-1, random_state=RANDOM_STATE, eval_metric="rmse",
        ),
        ("regression", "linear"): LinearRegression,
        ("regression", "random_forest"): lambda: RandomForestRegressor(
            n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
        ),
    }
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimators[(task, model_name)]()),
    ])


In [ ]:
def evaluate(task, model, X_valid, y_valid):
    predictions = model.predict(X_valid)

    if task == "classification":
        classes = list(model.named_steps["model"].classes_)
        if len(classes) == 2:
            average, pos_label = "binary", classes[-1]
        else:
            average, pos_label = "weighted", None
        metrics = {
            "accuracy": accuracy_score(y_valid, predictions),
            "precision": precision_score(y_valid, predictions, average=average, pos_label=pos_label, zero_division=0),
            "recall": recall_score(y_valid, predictions, average=average, pos_label=pos_label, zero_division=0),
            "f1": f1_score(y_valid, predictions, average=average, pos_label=pos_label, zero_division=0),
        }
        probabilities = model.predict_proba(X_valid)
        if len(classes) == 2:
            y_true_binary = (pd.Series(y_valid) == pos_label).astype(int)
            positive_probabilities = probabilities[:, classes.index(pos_label)]
            metrics["roc_auc"] = roc_auc_score(y_true_binary, positive_probabilities)
            metrics["pr_auc"] = average_precision_score(y_true_binary, positive_probabilities)
        else:
            metrics["roc_auc"] = roc_auc_score(y_valid, probabilities, multi_class="ovr", average="weighted", labels=classes)
    else:
        metrics = {
            "rmse": float(np.sqrt(mean_squared_error(y_valid, predictions))),
            "mae": mean_absolute_error(y_valid, predictions),
            "r2": r2_score(y_valid, predictions),
        }
    return metrics


## Load data and set up the task


In [ ]:
train = load_csv(TRAIN_CSV)

target = TARGET or train.columns[-1]
if target not in train.columns:
    raise ValueError(f"Target column '{target}' not found. Available: {list(train.columns)}")

id_column = ID_COLUMN
if id_column is None:
    candidates = [c for c in train.columns if c != target and re.match(r"(?:^|_)id$", c, flags=re.IGNORECASE)]
    id_column = candidates[0] if candidates else None

drop_columns = [target] + ([id_column] if id_column in train.columns else [])
X = train.drop(columns=drop_columns)
y = train[target]

task = detect_task(y)
model_name = MODEL or ("xgboost" if task == "classification" else "linear")

VALID_COMBINATIONS = {
    ("classification", "xgboost"),
    ("classification", "logistic"),
    ("classification", "random_forest"),
    ("regression", "xgboost"),
    ("regression", "linear"),
    ("regression", "random_forest"),
}
if (task, model_name) not in VALID_COMBINATIONS:
    raise ValueError(f"Model '{model_name}' cannot be used for a {task} target.")

print(f"Train data : {train.shape[0]} rows x {train.shape[1]} columns")
print(f"Target     : '{target}' ({task})")
print(f"ID column  : {id_column if id_column else 'none found - submissions will use row index'}")
print(f"Model      : {model_name}")
if task == "classification":
    print(f"Classes    : {y.value_counts().to_dict()}")


## Train and evaluate on a holdout split


In [ ]:
stratify = y if task == "classification" and y.value_counts().min() > 1 else None
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=stratify
)

model = build_pipeline(task, model_name, X, y_train)
model.fit(X_train, y_train)
metrics = evaluate(task, model, X_valid, y_valid)

print(f"Evaluation on {TEST_SIZE:.0%} holdout ({len(X_valid)} rows):")
for name, value in metrics.items():
    print(f"  {name:<9} {value:.4f}")


## Submission

`TEST_CSV` is currently `None`, so this cell reports evaluation results only. When the graded test csv is released: put its name in `TEST_CSV` in the configuration cell and re-run — the model is refit on all labelled rows and predictions are written to `submission.csv` (offered as a download in Colab).


In [ ]:
if TEST_CSV:
    test = load_csv(TEST_CSV)
    if target in test.columns:
        print(f"Note: test csv already contains '{target}' - dropping it before predicting.")
        test = test.drop(columns=[target])

    X_test = test[[c for c in test.columns if c != id_column]]
    final_model = build_pipeline(task, model_name, X, y)
    final_model.fit(X, y)
    test_predictions = final_model.predict(X_test)

    if id_column and id_column in test.columns:
        submission = pd.DataFrame({id_column: test[id_column], target: test_predictions})
    else:
        submission = pd.DataFrame({"row_id": test.index, target: test_predictions})
    submission.to_csv(SUBMISSION_OUT, index=False)
    print(f"Final model refit on all {len(X)} labelled rows.")
    print(f"Wrote {SUBMISSION_OUT} ({len(submission)} predictions, columns: {list(submission.columns)})")

    try:
        from google.colab import files
        files.download(SUBMISSION_OUT)
    except ImportError:
        pass
elif DEMO_SUBMISSION_ON_TRAIN:
    # experiment only - in-sample predictions on the training file, NOT a valid submission
    demo_model = build_pipeline(task, model_name, X, y)
    demo_model.fit(X, y)
    demo = pd.DataFrame({"row_id": train.index, target: demo_model.predict(X)})
    demo.to_csv(SUBMISSION_OUT, index=False)
    print(f"Wrote {SUBMISSION_OUT} from DEMO in-sample predictions ({len(demo)} rows) - do not submit this.")
else:
    print("No test set configured - evaluation only. Set TEST_CSV in the configuration cell and re-run to write", SUBMISSION_OUT)
